# Figure 3 a–c — two-dye mixtures and their misidentification rate (Ximea)

**Makes up:** Figure 3 a–c for four dye pairs (ATTO 520 + Rho6G, ATTO 565 + ATTO 594,
ATTO 565 + ATTO 633, ATTO 647N + QD800): per-molecule and per-frame histograms of photoelectrons
and relative pixel QE (`A_R`, `A_G`), how each molecule's colour narrows as its photoelectrons
accumulate, and the analytical misidentification rate against detected photoelectrons per molecule.

**Not here:** Figure 3 e–g (six-colour QDots) are made by `Figure_3_QDots.ipynb`.

**Data:** BioStudies `S-BIAD3210`, `2026_Multicolour_Paper/Ximea/DyeMixtures/`: one mixture per pair,
each a 4 × 4 grid of positions (one TIFF of ~1–2 GB plus one MicroManager metadata file each). The deposition holds raw data only, so this notebook fits it.

**Method** (from the analysis behind the published panels, `dye_discrimination/Dye_Mixture_Analysis_*`):
fit each position → AIM drift correction per position → DBSCAN clustering into molecules, after
quality cuts → the two dyes' mean colours from a 2-component GMM on the brightest molecules →
for each bin of accumulated photoelectrons, the two covariances fitted with those means fixed
(Tukey M-estimator) and the misidentification rate computed from the two Gaussians' overlap.
Shared code: `_multifov.py` (download / fit / cluster) and `_dye_pairs.py` (pairs, settings, analysis).

**Quality cuts** (all pairs): χ² < 3, colour errors < 1 (no cut on the background colour errors), > 50 detected photoelectrons,
PSF σ 75–250 nm; localisation error < 0.5 px (ATTO 565 + ATTO 594) or < 1 px (the others).
The published analysis used χ² < 2 and > 100 photons, and binned on `photons` (A_B + A_G + A_R,
which overcounts the detected photoelectrons 2–4×). Here the x axes are detected photoelectrons.

## FAST_MODE

`FAST_MODE = True` (default) downloads and fits one position per pair (4 TIFFs, ~8 GB, roughly
1 h of fitting): enough to run end to end, with ~1/16 of the molecules.
`FAST_MODE = False` uses all 64 positions (~130 GB, many hours of fitting; run in `tmux`/`screen`).

---
## 0 — Imports

In [ ]:
import sys
import types
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# _biostudies_download.py, _multifov.py and _dye_pairs.py live beside this notebook (notebooks/figures/).
sys.path.insert(0, str(Path.cwd().resolve()))
import _dye_pairs as dp

from pyS3M import IOFunctions
from pyS3M import sCMOSFunctions
from pyS3M import SR_Functions
from pyS3M import SM_extractionfunctions
from pyS3M import PlottingBase
from pyS3M.DriftCorrectionFunctions import Drift_Correction_Functions

IO = IOFunctions.IO_Functions()
sCMOS = sCMOSFunctions.sCMOS_Functions()
SupRes_F = SR_Functions.SuperRes_Functions()   # Ximea defaults (69 nm pixels)
SM_E = SM_extractionfunctions.extract_SMs()
drift_corrector = Drift_Correction_Functions()
plotter = PlottingBase.PublicationPlotter(dark_background=False)

print("Python:", sys.version)

---
## 1 — Configuration

In [ ]:
FAST_MODE = True   # True: one position per pair (~8 GB). False: all 64 positions (~130 GB, the published data).
REFIT = False      # True: fit again even if fits with the same files and settings exist.

REPO_ROOT = Path.cwd().resolve().parents[1]   # notebooks/figures -> repo root
CALIB_DIR = REPO_ROOT / "Camera_Calibrations" / "Ximea_Camera"
OUT_ROOT  = Path("outputs") / "Figure_3a-c_DyePairs"   # git-ignored (bare outputs/ rule)
FIG_DIR   = OUT_ROOT / ("fast" if FAST_MODE else "full")
# Point DATA_ROOT at a drive with room for the download. Each pair gets its own sub-folder,
# separately for each FAST_MODE setting: the fit reads every TIFF inside it.
DATA_ROOT = OUT_ROOT / ("data_fast" if FAST_MODE else "data_full")
FIG_DIR.mkdir(parents=True, exist_ok=True)
DATA_ROOT.mkdir(parents=True, exist_ok=True)

calib = dict(
    gain_map=IO.read_tiff(CALIB_DIR / "gain.tif"),
    offset_map=IO.read_tiff(CALIB_DIR / "offset.tif"),
    rqe=IO.read_tiff(CALIB_DIR / "rqe.tif"),
    read_noise=IO.read_tiff(CALIB_DIR / "readnoise.tif"),
    variance=IO.read_tiff(CALIB_DIR / "variance.tif"),
)

smoothing_function = types.SimpleNamespace()
smoothing_function.args = {"sigma": 1.5}
smoothing_function.extent = 1.5
smoothing_function.smoothing_function = sCMOS.gaussian_filter_stack
smoothing_function.data_arg = "image"

PAIRS = list(dp.PAIRS)
print(f"FAST_MODE  : {FAST_MODE}")
print(f"Data dir   : {DATA_ROOT.resolve()}")
print(f"Figures dir: {FIG_DIR.resolve()}")
print("Fit settings:", dp.FIT_PARAMS, "(PSF wavelength per pair)")

---
## 2 — Download, fit and drift-correct each pair

For each pair: download from BioStudies (with each position's metadata, which holds the camera
ROI the fit needs), fit every position separately (`fit_SM_data`, one `.h5` per TIFF), then AIM
drift correction per position. Re-running is safe: complete downloads are skipped, and fits are
reused unless the files or settings change (or `REFIT = True`).

In [ ]:
pair_files = {pair: dp.prepare_acquisition(pair, DATA_ROOT / pair, FAST_MODE, SupRes_F, calib,
                                           smoothing_function, drift_corrector, IO, refit=REFIT)
              for pair in PAIRS}
for pair, files in pair_files.items():
    print(f"{pair}: {len(files)} position(s)")

---
## 3 — Cluster localisations into molecules

DBSCAN, after the quality cuts (`dp.CLUSTER_PARAMS`): χ² < 3, colour errors < 1 (no cut on the background colour errors),
> 50 detected photoelectrons, PSF σ 75–250 nm, localisation error < 0.5 or 1 px; min. cluster size
5 (ATTO 565 + ATTO 594, ATTO 647N + QD800) or 2 (the others), as in the published analysis. A
molecule's colour is the inverse-variance weighted mean over its localisations.

`dp.photoelectron_accumulation` then follows each molecule frame by frame: its accumulated
photoelectrons and its running (inverse-variance weighted) colour.

In [ ]:
sm_db, sf_db, pa_db = {}, {}, {}
for pair in PAIRS:
    sm, sf = dp.cluster_sample(pair, SM_E, pair_files[pair], DATA_ROOT / pair)
    sm_db[pair], sf_db[pair] = IO.add_photoelectron_columns(sm), IO.add_photoelectron_columns(sf)
    pa_db[pair] = dp.photoelectron_accumulation(sf_db[pair], IO)
    print(f"{pair}: {len(sm):,} molecules, {len(sf):,} localisations, "
          f"median {sm_db[pair]['photoelectrons'].median():,.0f} photoelectrons per molecule")

---
## 4 — Per-molecule and per-frame histograms

For each pair: detected photoelectrons, and the relative pixel QE on the red (`A_R`) and green
(`A_G`) pixels; per molecule, then per localisation (frame).

In [ ]:
def histogram_row(db, pe_xlim, name):
    fig, axs = plotter.two_column_plot(nrows=1, ncols=3, height=1.6)
    pe = np.log10(db["photoelectrons"].to_numpy())
    plotter.histogram_plot(axs[0], pe, np.histogram_bin_edges(pe, "sqrt"),
                           xlabel=r"log$_{10}$(photoelectrons)", color="gray")
    plotter.histogram_plot(axs[1], db["A_R"], np.histogram_bin_edges(db["A_R"], "fd"),
                           xlabel="R/RGB", color="darkred")
    plotter.histogram_plot(axs[2], db["A_G"], np.histogram_bin_edges(db["A_G"], "fd"),
                           xlabel="G/RGB", color="darkgreen")
    axs[0].set_xlim(pe_xlim)
    for ax in axs[1:]:
        ax.set_xlim(0, 1)
    plt.tight_layout()
    plt.savefig(FIG_DIR / name, dpi=600, format="svg")
    plt.show()

for pair in PAIRS:
    print(pair, "per molecule")
    histogram_row(sm_db[pair], (2, 5.5), f"{pair}_Dye_Mixtures_perMolecule.svg")
    print(pair, "per frame")
    histogram_row(sf_db[pair], (1.5, 4), f"{pair}_Dye_Mixtures_perFrame.svg")

---
## 5 — Colour against accumulated photoelectrons

Each molecule's running colour, in 20 log-spaced bins of accumulated photoelectrons (10²–10⁵),
one molecule per bin (its first entry). Bright bins in light colours: the two dyes separate as
photoelectrons accumulate.

In [ ]:
PE_LEVELS = np.logspace(2, 5, 21)
level_colours = plt.cm.viridis(np.linspace(0, 1, len(PE_LEVELS) - 1))

for pair in PAIRS:
    pa = pa_db[pair]
    fig, axs = plotter.two_column_plot(nrows=1, ncols=3, height=1.6)
    for i, (lo, hi) in enumerate(zip(PE_LEVELS[:-1], PE_LEVELS[1:])):
        sub = pa[(pa["photoelectrons_accumulated"] >= lo) & (pa["photoelectrons_accumulated"] < hi)]
        sub = sub.groupby("molecular_index").first()
        if len(sub) < 10:
            continue
        pe = np.log10(sub["photoelectrons_accumulated"].to_numpy())
        plotter.histogram_plot(axs[0], pe, np.histogram_bin_edges(pe, "fd"), color=level_colours[i],
                               alpha=0.8, xlabel=r"log$_{10}$(accumulated photoelectrons)")
        plotter.histogram_plot(axs[1], sub["A_R"], np.histogram_bin_edges(sub["A_R"], "fd"),
                               color=level_colours[i], alpha=0.8, xlabel="R/RGB")
        plotter.histogram_plot(axs[2], sub["A_G"], np.histogram_bin_edges(sub["A_G"], "fd"),
                               color=level_colours[i], alpha=0.8, xlabel="G/RGB")
    axs[0].set_xlim(2, 5)
    for ax in axs[1:]:
        ax.set_xlim(0, 1)
    plt.tight_layout()
    plt.savefig(FIG_DIR / f"{pair}_Dye_Mixtures_vs_Photoelectrons.svg", dpi=600, format="svg")
    print(pair)
    plt.show()

---
## 6 — Misidentification rate against detected photoelectrons per molecule

The two mean colours come from a 2-component GMM fit to the bright molecules inside the pair's
colour box (`dp.REFERENCE_CUTS`, selected on `photons` as in the published analysis). Then, in each
bin of accumulated photoelectrons (`dp.PE_BINS`), each reference molecule's colour at its first
entry into the bin; the two covariances are fitted with the means fixed (Tukey M-estimator,
outliers down-weighted), and the misidentification rate is the overlap of the two Gaussians
(Monte Carlo, 10⁵ samples).

The panel is drawn as in the paper (`dp.error_rate_panel`: 63 × 40 mm axes, the paper's markers, fonts and 5 % line), with photoelectrons on the x axis instead of `photons`. Bins with fewer than 50 reference molecules, or no misidentified molecules (which a log axis can't show), are left out.

In [ ]:
summaries, fixed = {}, {}
for pair in PAIRS:
    refs = dp.reference_molecules(pair, sm_db[pair])
    means, ref_db, gmm = SM_E.extract_reference_means(refs, n_components=2, covariance_type="full",
                                                      fit_type="MLE", verbose=False)
    order = np.argsort(means[:, 0])   # lower A_R first, as the dyes in dp.PAIRS
    means, covs = means[order], gmm.covariances_[order]
    fixed[pair] = means
    summaries[pair] = SM_E.analyze_photon_dependent_misidentification_analytical(
        pa_db[pair], means, ref_db, dp.PE_BINS, reference_covariances=covs, use_earliest_entry=True,
        estimator_type="tukey", max_iter=20, n_mc_samples=100000, verbose=False)
    dyes = dp.PAIRS[pair][2]
    print(f"{pair}: {len(ref_db):,} reference molecules; "
          + "; ".join(f"{d} A_R {m[0]:.4f} A_G {m[1]:.4f}" for d, m in zip(dyes, means)))

In [ ]:
MIN_MOLECULES = 50   # bins with fewer reference molecules are not shown

# Figure 3c, as in the paper (dp.error_rate_panel): all four pairs on one 63 x 40 mm axis.
fig, ax = dp.error_rate_panel(summaries, min_molecules=MIN_MOLECULES)
plt.savefig(FIG_DIR / "DyePairs_Error_Rate.svg", dpi=600, format="svg")
plt.show()

# Each pair on its own, in the same style.
for pair in PAIRS:
    fig, ax = dp.error_rate_panel(summaries, pairs=[pair], min_molecules=MIN_MOLECULES)
    plt.savefig(FIG_DIR / f"{pair}_Dye_Error_Rate.svg", dpi=600, format="svg")
    print(pair)
    plt.show()

---
## 7 — Photoelectrons needed per pair

The fewest detected photoelectrons per molecule at which each pair's misidentification rate falls
below 5 %, 1 % and 0.1 % (the bin's lower edge; – if it never does in range). These are the
numbers quoted in the Results for Figure 3c.

In [ ]:
rows = {}
for pair in PAIRS:
    s = summaries[pair][summaries[pair]["n_molecules"] >= MIN_MOLECULES]
    rows[" + ".join(dp.PAIRS[pair][2])] = {
        f"< {level} %": (s.loc[100 * s["overall_error_rate"] < level, "photon_bin_min"].min())
        for level in (5, 1, 0.1)}
print(pd.DataFrame(rows).T.round(0).fillna("–"))